In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu
from scroutines import lmm

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm

In [2]:
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_nrdr_lmm'
!mkdir -p $outfigdir

In [3]:
adata = sc.read("/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_cheng22_astro.h5ad")
adata

AnnData object with n_obs × n_vars = 19330 × 16340
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'
    layers: 'norm'

In [4]:
np.unique(adata.obs.Sample)


array(['P14_1a', 'P14_1b', 'P14_2a', 'P14_2b', 'P17_1a', 'P17_1b',
       'P17_2a', 'P17_2b', 'P21_1a', 'P21_1b', 'P21_2a', 'P21_2b',
       'P28_1a', 'P28_1b', 'P28_2a', 'P28_2b', 'P28_dl_1a', 'P28_dl_1b',
       'P28_dl_2a', 'P28_dl_2b', 'P28_dr_1a', 'P28_dr_1b', 'P28_dr_3a',
       'P28_dr_3b', 'P38_1a', 'P38_2a', 'P38_2b', 'P38_dr_1a',
       'P38_dr_1b', 'P38_dr_2a', 'P38_dr_2b', 'P8_1a', 'P8_1b', 'P8_2a',
       'P8_2b'], dtype=object)

In [5]:
# meta = meta[meta['Age'].isin(['P28_nr', 'P28_dr'])]

In [6]:
# remove mitocondria genes
adata = adata[:,~adata.var.index.str.contains(r'^mt-')]

# remove sex genes
sex_genes = ["Xist", "Uty", "Eif2s3y", "Kdm5d", "Ddx3y"]
adata = adata[:,[g for g in adata.var.index if g not in sex_genes]]

# filter genes
cond = np.ravel((adata.X>0).sum(axis=0)) > 10 # expressed in more than 10 cells
adata = adata[:,cond].copy()
# genes = adata.var.index.values

adata

AnnData object with n_obs × n_vars = 19330 × 16323
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden'
    var: 'feature_types'
    layers: 'norm'

In [7]:
np.array(natsorted(np.unique(adata.obs['Age'].values)))

array(['P8', 'P14', 'P17', 'P21', 'P28', 'P28_dl', 'P28_dr', 'P38',
       'P38_dr'], dtype='<U6')

In [8]:
cell_abundances = adata.obs.groupby(['Subclass', 'Age']).size().unstack()
cell_abundances
# value_counts()

Age,P8,P14,P17,P21,P28,P28_dl,P28_dr,P38,P38_dr
Subclass,,,,,,,,,
Astro,1411,2808,2821,2516,2168,2077,1960,1485,2084


In [9]:
cell_abundances = adata.obs.groupby(['Age', 'Sample']).size().unstack()
cell_abundances

Sample,P8_1a,P8_1b,P8_2a,P8_2b,P14_1a,P14_1b,P14_2a,P14_2b,P17_1a,P17_1b,...,P28_dr_1b,P28_dr_3a,P28_dr_3b,P38_1a,P38_2a,P38_2b,P38_dr_1a,P38_dr_1b,P38_dr_2a,P38_dr_2b
Age,,,,,,,,,,,,,,,,,,,,,
P8,325,248,399,439,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
P14,0,0,0,0,835,804,579,590,0,0,...,0,0,0,0,0,0,0,0,0,0
P17,0,0,0,0,0,0,0,0,726,754,...,0,0,0,0,0,0,0,0,0,0
P21,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
P28,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
P28_dl,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
P28_dr,0,0,0,0,0,0,0,0,0,0,...,350,635,670,0,0,0,0,0,0,0
P38,0,0,0,0,0,0,0,0,0,0,...,0,0,0,280,604,601,0,0,0,0
P38_dr,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,656,318,539,571


In [10]:
# adata.obs['cond'] = adata.obs['cond'].apply(lambda x: x.replace('NR', ""))

# sample_labels = adata.obs['Sample'].values
# time_labels = [s[:-1].replace('DR', '') for s in sample_labels]

# adata.obs['sample'] = sample_labels #
# adata.obs['time']   = time_labels

# uniq_samples = natsorted(np.unique(sample_labels))
# nr_samples = [s for s in uniq_samples if "DR" not in s]
# dr_samples = [s for s in uniq_samples if "DR" in s]

# uniq_conds = np.array(natsorted(np.unique(adata.obs['cond'].values)))

# print(uniq_conds)



In [11]:
def streamline_age(age):
    """
    """
    tags = age.split('_')
    age = tags[0]
    
    if len(tags) == 1:
        return age, "NR"
    elif len(tags) == 2:
        return age, tags[1].upper()
    
adata.obs['Time'] = adata.obs['Age'].apply(lambda x: streamline_age(x)[0])
adata.obs['Light'] = adata.obs['Age'].apply(lambda x: streamline_age(x)[1])
adata.obs

,Age,Doublet,Doublet Score,n_counts,n_genes,percent_mito,sample,Type,Subclass,Class,Sample,total_counts,pct_counts_mt,n_genes_by_counts,total_counts_mt,Doublet?,Study,Type_leiden,Time,Light
AACCAACGTGCAGGAT-1-P8_1a-2022 RNA-1-0,P8,False,0.015945,7199.0,2704.0,0.000833,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,7199.0,0.000833,NaN,NaN,NaN,2022 RNA,Astro_A,P8,NR
AACCACACAGACACCC-1-P8_1a-2022 RNA-1-0,P8,False,0.008629,14131.0,4076.0,0.000142,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,14131.0,0.000142,NaN,NaN,NaN,2022 RNA,Astro_A,P8,NR
AAGACAAAGCAAACAT-1-P8_1a-2022 RNA-1-0,P8,False,0.012605,4837.0,2095.0,0.001034,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,4837.0,0.001034,NaN,NaN,NaN,2022 RNA,Astro_A,P8,NR
AAGTGAACAGGCATGA-1-P8_1a-2022 RNA-1-0,P8,False,0.016672,6017.0,2496.0,0.000499,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,6017.0,0.000499,NaN,NaN,NaN,2022 RNA,Astro_A,P8,NR
ACACAGTAGGGAGGCA-1-P8_1a-2022 RNA-1-0,P8,False,0.008117,7615.0,2792.0,0.000131,P8_1a,Astro_A,Astro,Non-neurons,P8_1a,7615.0,0.000131,NaN,NaN,NaN,2022 RNA,Astro_A,P8,NR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ATCGATGCATGCCATA-1-P38_dr_2b-5,P38_dr,False,0.117371,3916.0,2040.0,0.000766,P38_dr_2b,Astro_A,Astro,Non-neuron,P38_dr_2b,NaN,NaN,NaN,NaN,NaN,2022 RNA,NaN,P38,DR
TCTACCGCATCGGCCA-1-P38_dr_2b-5,P38_dr,False,0.011913,4748.0,2315.0,0.000632,P38_dr_2b,Astro_A,Astro,Non-neuron,P38_dr_2b,NaN,NaN,NaN,NaN,NaN,2022 RNA,NaN,P38,DR
GGGAAGTTCCATGATG-1-P38_dr_2b-5,P38_dr,False,0.117371,4978.0,2396.0,0.003213,P38_dr_2b,Astro_A,Astro,Non-neuron,P38_dr_2b,NaN,NaN,NaN,NaN,NaN,2022 RNA,NaN,P38,DR
AAGCGTTGTGTTGACT-1-P38_dr_2b-5,P38_dr,False,0.013906,2610.0,1562.0,0.001148,P38_dr_2b,Astro_A,Astro,Non-neuron,P38_dr_2b,NaN,NaN,NaN,NaN,NaN,2022 RNA,NaN,P38,DR


In [ ]:
%%time

tag = 'd251121'
output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_astro_cheng22_{tag}.h5ad')

exp_conds = ['P28', 'P28_dr', 'P38', 'P38_dr']
obs_fixed1 = 'Time'
obs_fixed2 = 'Light'
obs_random = 'Sample'


offset = 1e-2
scale = 1e4

adatasub = adata[adata.obs['Age'].isin(exp_conds)]
# ### test
# adatasub = adatasub[:,:20]
# ### test

genes = adatasub.var.index.values 

obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
obs = obs.dropna()

adatasub = adatasub[obs.index]

# mat (CP10k norm)
mat = np.array(adatasub.X.todense())/adatasub.obs['n_counts'].values.reshape(-1,1)*scale

res = lmm.run_lmm_two_fixed(mat, genes, obs, obs_fixed1, obs_fixed2, obs_random, output=output, offset=offset)

(7697, 16323) (7697, 3)
(7697, 16321) (7697, 3)
(7697, 11147) (7697, 3)


100% 11128/11147 [1:13:09<00:07,  2.58it/s]

In [ ]:
print(output)

In [ ]:
sc.read(output)

In [ ]:
res

In [ ]:
res.layers['pval']

In [ ]:
res.var

In [ ]:
res.var